# Queue-Size-Prediction-LSTM

#### Imports and config

In [32]:
%matplotlib inline
import tensorflow as tf
import numpy as np
import pandas as pd
from pandas.io.json import json_normalize
import math
import pickle
import json
import datetime

In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from sklearn.preprocessing import MinMaxScaler

## Preprocessing

Open the file and read content to a list

In [5]:
with open('20200501-queues-without-items-1219-till-0520.json', 'r') as f:
    res = list()
    for line in f.readlines():
        res.append(json.loads(line))

Build a pandas dataframe from list

In [6]:
df = pd.DataFrame(res)

Narrow it down to the _source column

In [7]:
df = df['_source']

Normalize Dataframe

In [8]:
df = json_normalize(df)

Narrow it down to the products queue

In [9]:
df = df[df['name'] == "products"]

Set the timestamp date as index

In [10]:
df.index = df["timestamp"]

Convert the index to datetimeformat

In [11]:
df.index = pd.to_datetime(df.index, format='%Y-%m-%dT%H:%M:%S.%f%z').sort_values()

Drop unnecessary columns

In [12]:
df.drop(columns=['timestamp', 'name', 'tier', 'querytime'], inplace=True)

Resample Data to 15min Interval 

In [13]:
df = df.resample('15T').mean()

Fill all NA values with 0 

In [14]:
df = df.fillna(0)

Final Dataframe

In [15]:
df.head()

,size
timestamp,
2019-12-05 15:30:00+00:00,108.000000
2019-12-05 15:45:00+00:00,121.400000
2019-12-05 16:00:00+00:00,99.933333
2019-12-05 16:15:00+00:00,95.066667
2019-12-05 16:30:00+00:00,129.933333


In [16]:
df.tail()

,size
timestamp,
2020-05-03 20:00:00+00:00,28955.500000
2020-05-03 20:15:00+00:00,28884.133333
2020-05-03 20:30:00+00:00,28805.433333
2020-05-03 20:45:00+00:00,28731.133333
2020-05-03 21:00:00+00:00,28697.500000


### Pickle for faster testing

Save the final Dataframe as pickle file

In [17]:
pickle.dump(df, open("woq_df_15min.p","wb"))

Load Pickle Dataframe

In [18]:
df = pickle.load(open("woq_df_15min.p","rb"))

In [19]:
df.head()

,size
timestamp,
2019-12-05 15:30:00+00:00,108.000000
2019-12-05 15:45:00+00:00,121.400000
2019-12-05 16:00:00+00:00,99.933333
2019-12-05 16:15:00+00:00,95.066667
2019-12-05 16:30:00+00:00,129.933333


Get only the values from the Dataframe

In [20]:
train = df.values

Scale the values

In [21]:
scaler = MinMaxScaler()
scaler.fit(train)
train = scaler.transform(train)

Define Generator

In [22]:
n_input = 96 #input length - 1day = 96*15min
n_features = 1 #size as the only feature
n_units = 20 #number of neurons used by the network
generator = TimeseriesGenerator(train, train, length=n_input, batch_size=192)

Size of the generator

In [23]:
len(generator)

75

Structure of the generator

In [30]:
for i in range(len(generator)):
    x, y = generator[i]
    print('%s => %s' % (len(x), len(y)))
    break

192 => 192


Build the model

In [31]:
model = Sequential()
model.add(LSTM(n_units, activation='tanh', input_shape=(n_input, n_features)))
model.add(Dropout(0.2))
model.add(Dense(1))
model.compile(loss='mse', optimizer='adam', metrics=['mse'])
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
lstm (LSTM)                  (None, 20)                1760      
_________________________________________________________________
dropout (Dropout)            (None, 20)                0         
_________________________________________________________________
dense (Dense)                (None, 1)                 21        
Total params: 1,781
Trainable params: 1,781
Non-trainable params: 0
_________________________________________________________________


Train the model

In [63]:
history = model.fit_generator(generator,epochs=5,verbose=1)

Epoch 1/5
75/75 [==============================] - 231s 3s/step - loss: 0.0295 - mse: 0.0283
Epoch 2/5
75/75 [==============================] - 237s 3s/step - loss: 0.0082 - mse: 0.0081
Epoch 3/5
75/75 [==============================] - 230s 3s/step - loss: 0.0016 - mse: 0.0016
Epoch 4/5
75/75 [==============================] - 234s 3s/step - loss: 0.0014 - mse: 0.0014
Epoch 5/5
75/75 [==============================] - 233s 3s/step - loss: 0.0013 - mse: 0.0012


Save the model

In [64]:
model.save("model_lstm_15min_woi.h5")

Make Prediction

In [65]:
pred_list = []

batch = train[-n_input:].reshape((1, n_input, n_features)) #first batch is the last day in

# for 96 steps, always predict 1 step ahead and add that to the pred_list and update the batch with it
for i in range(n_input):   
    pred_list.append(model.predict(batch)[0]) 
    batch = np.append(batch[:,1:,:],[[pred_list[i]]],axis=1)

In [67]:
pred_list

[array([0.47482497], dtype=float32),
 array([0.4735257], dtype=float32),
 array([0.47204342], dtype=float32),
 array([0.47047248], dtype=float32),
 array([0.4688546], dtype=float32),
 array([0.4672124], dtype=float32),
 array([0.46555746], dtype=float32),
 array([0.46389556], dtype=float32),
 array([0.4622297], dtype=float32),
 array([0.46056136], dtype=float32),
 array([0.45889124], dtype=float32),
 array([0.45721954], dtype=float32),
 array([0.45554647], dtype=float32),
 array([0.45387188], dtype=float32),
 array([0.45219573], dtype=float32),
 array([0.45051795], dtype=float32),
 array([0.44883835], dtype=float32),
 array([0.44715676], dtype=float32),
 array([0.44547305], dtype=float32),
 array([0.443787], dtype=float32),
 array([0.44209853], dtype=float32),
 array([0.44040748], dtype=float32),
 array([0.4387136], dtype=float32),
 array([0.43701676], dtype=float32),
 array([0.43531677], dtype=float32),
 array([0.43361357], dtype=float32),
 array([0.43190682], dtype=float32),
 array([

Rescale the prediction values

In [95]:
pred = np.array(scaler.inverse_transform(pred_list)).reshape(-1)

In [96]:
pred[:1]

array([28346.55975805])

Convert to integer numbers

In [98]:
pred = [int(x) for x in pred]

In [87]:
pred.shape

(96,)

Build the Prediction Dataframe

In [104]:
time_stamps = pd.date_range(start=df.index[-1]+datetime.timedelta(minutes=15), periods=96, freq='15min',tz='utc')

In [105]:
d = {'timestamp': time_stamps, 'size': pred}
pred_df = pd.DataFrame(data=d)

In [106]:
pred_df

,timestamp,size
0,2020-05-03 21:15:00+00:00,28346
1,2020-05-03 21:30:00+00:00,28268
2,2020-05-03 21:45:00+00:00,28180
3,2020-05-03 22:00:00+00:00,28086
4,2020-05-03 22:15:00+00:00,27990
...,...,...
91,2020-05-04 20:00:00+00:00,18305
92,2020-05-04 20:15:00+00:00,18172
93,2020-05-04 20:30:00+00:00,18039
94,2020-05-04 20:45:00+00:00,17905
